# Experiment 5 — Noise Analysis and Error Mitigation
Models realistic quantum noise and recovers ideal results using **MEM** and **ZNE**.

In [ ]:
!pip install qiskit qiskit-aer matplotlib numpy -q

## Imports

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, ReadoutError
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt

## Step 1 — Test Circuit (Bell State)
Ideal output: 50% |00⟩ and 50% |11⟩. Any deviation = effect of noise.

In [ ]:
def bell_circuit():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

test_circuit = bell_circuit()
print(test_circuit.draw('text'))

## Step 2 — Build Realistic Noise Model

In [ ]:
def make_noise_model():
    model = NoiseModel()

    model.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['h', 'x', 'z'])
    model.add_all_qubit_quantum_error(depolarizing_error(0.01,  2), ['cx'])

    t1, t2, gate_time = 50_000, 70_000, 100
    single_gate_noise = thermal_relaxation_error(t1, t2, gate_time)
    two_gate_noise    = single_gate_noise.expand(single_gate_noise)
    model.add_all_qubit_quantum_error(single_gate_noise, ['h', 'x'])
    model.add_all_qubit_quantum_error(two_gate_noise,    ['cx'])

    model.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.02, 0.98]]))
    return model

noise_model = make_noise_model()
print("Noise model ready.")

## Step 3 — Ideal vs Noisy Comparison

In [ ]:
total_shots  = 4096
ideal_sim    = AerSimulator()
noisy_sim    = AerSimulator(noise_model=noise_model)

ideal_counts = ideal_sim.run(transpile(test_circuit, ideal_sim), shots=total_shots).result().get_counts()
noisy_counts = noisy_sim.run(transpile(test_circuit, noisy_sim), shots=total_shots).result().get_counts()

print("Ideal counts:", ideal_counts)
print("Noisy counts:", noisy_counts)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_histogram(ideal_counts, ax=axes[0], title="Ideal")
plot_histogram(noisy_counts, ax=axes[1], title="Noisy")
plt.tight_layout()
plt.show()

## Step 4 — Measurement Error Mitigation (MEM)
Build a calibration matrix from known input states, then invert it to correct noisy results.

In [ ]:
def apply_mem(num_qubits, noisy_simulator, raw_counts, shots):
    size = 2 ** num_qubits

    calibration_circuits = []
    for state_idx in range(size):
        bits = format(state_idx, f'0{num_qubits}b')
        cal  = QuantumCircuit(num_qubits, num_qubits)
        for i, b in enumerate(reversed(bits)):
            if b == '1':
                cal.x(i)
        cal.measure(range(num_qubits), range(num_qubits))
        calibration_circuits.append(cal)

    cal_result = noisy_simulator.run(transpile(calibration_circuits, noisy_simulator), shots=shots).result()

    calibration_matrix = np.zeros((size, size))
    for i, cal in enumerate(calibration_circuits):
        for key, val in cal_result.get_counts(cal).items():
            calibration_matrix[int(key, 2)][i] = val / shots

    raw_vector     = np.array([raw_counts.get(format(i, '02b'), 0) / shots for i in range(size)])
    corrected      = np.maximum(np.linalg.pinv(calibration_matrix) @ raw_vector, 0)
    corrected     /= corrected.sum()
    return corrected

mem_result = apply_mem(2, noisy_sim, noisy_counts, total_shots)

print("\nMEM Correction Results:")
for i in range(4):
    label = format(i, '02b')
    print(f"  |{label}>  ideal={ideal_counts.get(label,0)/total_shots:.3f}"
          f"  noisy={noisy_counts.get(label,0)/total_shots:.3f}"
          f"  MEM={mem_result[i]:.3f}")

## Step 5 — Zero Noise Extrapolation (ZNE)
Artificially scale up noise, then extrapolate back to zero noise.

In [ ]:
def scale_noise(original_circuit, scale):
    if scale == 1:
        return original_circuit
    base = original_circuit.remove_final_measurements(inplace=False)
    scaled = QuantumCircuit(*base.qregs, *base.cregs)
    for instruction in base.data:
        gate  = instruction.operation
        qargs = instruction.qubits
        scaled.append(gate, qargs)
        for _ in range(scale - 1):
            scaled.append(gate.inverse(), qargs)
            scaled.append(gate, qargs)
    scaled.measure_all()
    return scaled

def zz_value(counts, shots):
    return ((counts.get('00', 0) + counts.get('11', 0)) -
            (counts.get('01', 0) + counts.get('10', 0))) / shots

noise_levels  = [1, 3, 5]
zz_readings   = []

for level in noise_levels:
    scaled_circuit = scale_noise(test_circuit, level)
    run_result     = noisy_sim.run(transpile(scaled_circuit, noisy_sim), shots=total_shots).result()
    reading        = zz_value(run_result.get_counts(), total_shots)
    zz_readings.append(reading)
    print(f"Noise scale {level}x  →  <ZZ> = {reading:.4f}")

fit_line     = np.polyfit(noise_levels, zz_readings, 1)
zne_estimate = np.polyval(fit_line, 0)
print(f"\nZNE extrapolated result: <ZZ> = {zne_estimate:.4f}")
print(f"Ideal result:            <ZZ> = 1.0000")

x_range = np.linspace(0, 6, 100)
plt.figure(figsize=(6, 4))
plt.plot(x_range, np.polyval(fit_line, x_range), 'k--', label='Linear fit')
plt.scatter(noise_levels, zz_readings, color='black', s=80, zorder=5)
plt.scatter([0], [zne_estimate], color='black', s=150, marker='*', zorder=6,
            label=f'ZNE = {zne_estimate:.3f}')
plt.axhline(1.0, color='gray', linestyle=':', label='Ideal = 1.0')
plt.xlabel("Noise Scale Factor")
plt.ylabel("<ZZ>")
plt.title("Zero Noise Extrapolation (ZNE)")
plt.legend()
plt.tight_layout()
plt.show()